# Task 2: Bayesian Change Point Detection for Brent Oil Prices

## Objective
Apply Bayesian change point detection to identify and quantify structural breaks in Brent oil prices, and associate these changes with major geopolitical and economic events.

## Methodology
- Use PyMC for Bayesian inference
- Model discrete change points with switching means
- Perform MCMC sampling for posterior inference
- Associate detected change points with historical events

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pymc as pm
import arviz as az
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Set plotting style
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")

In [ ]:
# Load and prepare data\nprint(\"Loading Brent oil price data...\")\n\n# Load price data\nprice_df = pd.read_csv('../Data/raw/BrentOilPrices.csv')\nprint(f\"Price data shape: {price_df.shape}\")\nprint(f\"Price data columns: {price_df.columns.tolist()}\")\nprint(f\"First few rows:\\n{price_df.head()}\")\n\n# Convert date column with multiple format handling\ndef parse_dates(date_str):\n    try:\n        # Try format '20-May-1987'\n        return pd.to_datetime(date_str, format='%d-%b-%Y')\n    except:\n        try:\n            # Try format 'May 20, 1987'\n            return pd.to_datetime(date_str)\n        except:\n            return pd.NaT\n\nprice_df['Date'] = price_df['Date'].apply(parse_dates)\nprice_df = price_df.dropna(subset=['Date'])\nprice_df = price_df.sort_values('Date').reset_index(drop=True)\n\nprint(f\"\\nAfter date cleaning:\")\nprint(f\"Date range: {price_df['Date'].min()} to {price_df['Date'].max()}\")\nprint(f\"Total observations: {len(price_df)}\")\n\n# Load event data\nevents_df = pd.read_csv('../Data/events/oil_market_events_fixed.csv')\nevents_df['Date'] = pd.to_datetime(events_df['Date'])\nprint(f\"\\nEvent data shape: {events_df.shape}\")\nprint(f\"Event types: {events_df['Event_Type'].value_counts()}\")\nprint(f\"\\nSample events:\\n{events_df[['Date', 'Event', 'Event_Type', 'Severity']].head()}\")

In [ ]:
# Plot raw prices and log returns\nfig, axes = plt.subplots(2, 2, figsize=(15, 10))\n\n# Raw price series\naxes[0, 0].plot(price_df['Date'], price_df['Price'], linewidth=1)\naxes[0, 0].set_title('Brent Oil Price Series (1987-2022)', fontsize=14, fontweight='bold')\naxes[0, 0].set_xlabel('Date')\naxes[0, 0].set_ylabel('Price (USD)')\naxes[0, 0].grid(True, alpha=0.3)\n\n# Add major events to price plot\nfor _, event in events_df.iterrows():\n    event_date = event['Date']\n    if price_df['Date'].min() <= event_date <= price_df['Date'].max():\n        axes[0, 0].axvline(event_date, color='red', alpha=0.3, linestyle='--')\n        axes[0, 0].text(event_date, price_df['Price'].max() * 0.9, \n                       event['Event'][:15] + '...', rotation=90, \n                       fontsize=8, color='red')\n\n# Log returns\nprice_df['Log_Return'] = np.log(price_df['Price']).diff()\naxes[0, 1].plot(price_df['Date'], price_df['Log_Return'], linewidth=0.5, alpha=0.7)\naxes[0, 1].set_title('Log Returns of Brent Oil Prices', fontsize=14, fontweight='bold')\naxes[0, 1].set_xlabel('Date')\naxes[0, 1].set_ylabel('Log Return')\naxes[0, 1].grid(True, alpha=0.3)\n\n# Volatility clustering (rolling standard deviation)\nrolling_vol = price_df['Log_Return'].rolling(window=30).std()\naxes[1, 0].plot(price_df['Date'], rolling_vol, linewidth=1, color='orange')\naxes[1, 0].set_title('30-Day Rolling Volatility', fontsize=14, fontweight='bold')\naxes[1, 0].set_xlabel('Date')\naxes[1, 0].set_ylabel('Volatility')\naxes[1, 0].grid(True, alpha=0.3)\n\n# Distribution of returns\naxes[1, 1].hist(price_df['Log_Return'].dropna(), bins=100, alpha=0.7, color='skyblue', edgecolor='black')\naxes[1, 1].set_title('Distribution of Log Returns', fontsize=14, fontweight='bold')\naxes[1, 1].set_xlabel('Log Return')\naxes[1, 1].set_ylabel('Frequency')\naxes[1, 1].grid(True, alpha=0.3)\n\n# Add normal distribution overlay\nfrom scipy import stats\nmu, sigma = stats.norm.fit(price_df['Log_Return'].dropna())\nx = np.linspace(price_df['Log_Return'].min(), price_df['Log_Return'].max(), 100)\naxes[1, 1].plot(x, stats.norm.pdf(x, mu, sigma) * len(price_df['Log_Return'].dropna()) * \n               (price_df['Log_Return'].max() - price_df['Log_Return'].min()) / 100, \n               'r-', linewidth=2, label=f'Normal(μ={mu:.4f}, σ={sigma:.4f})')\naxes[1, 1].legend()\n\nplt.tight_layout()\nplt.savefig('../reports/task2_price_analysis.png', dpi=300, bbox_inches='tight')\nplt.show()\n\n# Summary statistics\nprint(\"\\nSummary Statistics:\")\nprint(f\"Price statistics:\")\nprint(price_df['Price'].describe())\nprint(f\"\\nLog Return statistics:\")\nprint(price_df['Log_Return'].describe())\nprint(f\"\\nVolatility clustering evidence:\")\nprint(f\"Autocorrelation of squared returns (lag 1): {price_df['Log_Return'].dropna().pow(2).autocorr(lag=1):.4f}\")

In [ ]:
# Prepare data for change point analysis\n# Use log returns for better statistical properties\nreturns_data = price_df['Log_Return'].dropna().values\nn_obs = len(returns_data)\n\nprint(f\"Preparing change point analysis with {n_obs} observations\")\nprint(f\"Date range for analysis: {price_df['Date'].iloc[1]} to {price_df['Date'].iloc[-1]}\")\n\n# Build Bayesian change point model\nprint(\"\\nBuilding Bayesian change point model...\")\n\nwith pm.Model() as change_point_model:\n    # Prior for change point location (discrete uniform over the time series)\n    tau = pm.DiscreteUniform('tau', lower=0, upper=n_obs-1)\n    \n    # Priors for means before and after change point\n    mu_1 = pm.Normal('mu_1', mu=0, sigma=0.01)  # Mean before change point\n    mu_2 = pm.Normal('mu_2', mu=0, sigma=0.01)  # Mean after change point\n    \n    # Prior for standard deviation (assumed same before and after)\n    sigma = pm.HalfCauchy('sigma', beta=0.01)\n    \n    # Switch function to select appropriate mean based on tau\n    mu = pm.math.switch(tau >= np.arange(n_obs), mu_1, mu_2)\n    \n    # Likelihood\n    likelihood = pm.Normal('y', mu=mu, sigma=sigma, observed=returns_data)\n    \n    # Print model summary\n    print(\"Model structure:\")\n    print(f\"- Change point tau: Uniform(0, {n_obs-1})\")\n    print(f\"- Mean before: Normal(0, 0.01)\")\n    print(f\"- Mean after: Normal(0, 0.01)\")\n    print(f\"- Standard deviation: HalfCauchy(0, 0.01)\")\n    print(f\"- Likelihood: Normal(mu, sigma)\")

In [ ]:
# Run MCMC sampling\nprint(\"Running MCMC sampling...\")\n\nwith change_point_model:\n    # Use Metropolis sampler for discrete parameter (tau)\n    step = pm.Metropolis()\n    \n    # Run sampling\n    trace = pm.sample(\n        draws=10000, \n        tune=2000, \n        step=step,\n        chains=4,\n        random_seed=42,\n        progressbar=True,\n        return_inferencedata=True\n    )\n\nprint(\"\\nMCMC sampling completed!\")\nprint(f\"Total samples: {trace.posterior.dims['draw'] * trace.posterior.dims['chain']}\")\nprint(f\"Chains: {trace.posterior.dims['chain']}\")\nprint(f\"Draws per chain: {trace.posterior.dims['draw']}\")

In [ ]:
# Model diagnostics and convergence checks\nprint(\"Model Diagnostics:\")\nprint(\"==================\")\n\n# Check convergence using R-hat statistics\nprint(\"\\nConvergence diagnostics (R-hat):\")\nprint(az.summary(trace, var_names=['tau', 'mu_1', 'mu_2', 'sigma'], round_to=4))\n\n# Effective sample sizes\nprint(\"\\nEffective sample sizes:\")\nprint(az.ess(trace, var_names=['tau', 'mu_1', 'mu_2', 'sigma']))\n\n# Trace plots\naz.plot_trace(trace, var_names=['tau', 'mu_1', 'mu_2', 'sigma'])\nplt.tight_layout()\nplt.savefig('../reports/task2_trace_plots.png', dpi=300, bbox_inches='tight')\nplt.show()\n\n# Posterior distributions\nfig, axes = plt.subplots(2, 2, figsize=(12, 8))\n\n# Change point posterior\naz.plot_posterior(trace, var_names=['tau'], ax=axes[0, 0])\naxes[0, 0].set_title('Posterior Distribution of Change Point (tau)')\n\n# Means posterior\naz.plot_posterior(trace, var_names=['mu_1'], ax=axes[0, 1])\naxes[0, 1].set_title('Posterior Distribution of Mean Before Change Point')\n\naz.plot_posterior(trace, var_names=['mu_2'], ax=axes[1, 0])\naxes[1, 0].set_title('Posterior Distribution of Mean After Change Point')\n\naz.plot_posterior(trace, var_names=['sigma'], ax=axes[1, 1])\naxes[1, 1].set_title('Posterior Distribution of Standard Deviation')\n\nplt.tight_layout()\nplt.savefig('../reports/task2_posterior_distributions.png', dpi=300, bbox_inches='tight')\nplt.show()

In [ ]:
# Interpret change point results\nprint(\"Change Point Analysis Results:\")\nprint(\"===============================\")\n\n# Extract posterior samples\ntau_samples = trace.posterior['tau'].values.flatten()\nmu_1_samples = trace.posterior['mu_1'].values.flatten()\nmu_2_samples = trace.posterior['mu_2'].values.flatten()\nsigma_samples = trace.posterior['sigma'].values.flatten()\n\n# Calculate summary statistics\ntau_mean = np.mean(tau_samples)\ntau_median = np.median(tau_samples)\ntau_hpd = az.hdi(tau_samples, hdi_prob=0.95)\n\nmu_1_mean = np.mean(mu_1_samples)\nmu_2_mean = np.mean(mu_2_samples)\n\nprint(f\"\\nChange Point Location:\")\nprint(f\"- Mean: {tau_mean:.1f} (day index)\")\nprint(f\"- Median: {tau_median:.1f} (day index)\")\nprint(f\"- 95% HDI: [{tau_hpd[0]:.1f}, {tau_hpd[1]:.1f}]\")\n\n# Convert to actual dates\nchange_point_date = price_df['Date'].iloc[int(tau_median) + 1]  # +1 because we used returns\nprint(f\"- Estimated Date: {change_point_date.strftime('%Y-%m-%d')}\")\n\nprint(f\"\\nMean Before Change Point: {mu_1_mean:.6f}\")\nprint(f\"Mean After Change Point: {mu_2_mean:.6f}\")\nprint(f\"Difference: {mu_2_mean - mu_1_mean:.6f}\")\nprint(f\"Percent Change: {((mu_2_mean - mu_1_mean) / abs(mu_1_mean) * 100):.2f}%\")\n\n# Find events near the change point\nevent_window = pd.Timedelta(days=60)  # Look 60 days before and after\nnearby_events = events_df[\n    (events_df['Date'] >= change_point_date - event_window) &\n    (events_df['Date'] <= change_point_date + event_window)\n].sort_values('Date')\n\nprint(f\"\\nEvents within ±60 days of detected change point:\")\nif len(nearby_events) > 0:\n    for _, event in nearby_events.iterrows():\n        days_diff = (event['Date'] - change_point_date).days\n        direction = \"before\" if days_diff < 0 else \"after\" if days_diff > 0 else \"on\"\n        print(f\"- {event['Date'].strftime('%Y-%m-%d')} ({abs(days_diff)} days {direction}): {event['Event']} ({event['Event_Type']}, Severity: {event['Severity']})\")\nelse:\n    print(\"No major events found within ±60 days of the change point.\")\n\n# Visualize change point on the data\nfig, axes = plt.subplots(2, 1, figsize=(15, 10))\n\n# Plot with change point on returns\naxes[0].plot(price_df['Date'][1:], returns_data, linewidth=0.5, alpha=0.7, label='Log Returns')\naxes[0].axvline(change_point_date, color='red', linestyle='--', linewidth=2, label=f'Detected Change Point: {change_point_date.strftime(\"%Y-%m-%d\")}')\naxes[0].set_title('Log Returns with Detected Change Point', fontsize=14, fontweight='bold')\naxes[0].set_xlabel('Date')\naxes[0].set_ylabel('Log Return')\naxes[0].legend()\naxes[0].grid(True, alpha=0.3)\n\n# Add nearby events\nfor _, event in nearby_events.iterrows():\n    axes[0].axvline(event['Date'], color='orange', alpha=0.5, linestyle=':')\n    axes[0].text(event['Date'], returns_data.max() * 0.8, event['Event'][:20] + '...', \n                rotation=90, fontsize=8, color='orange')\n\n# Plot cumulative returns with change point\ncumulative_returns = np.cumsum(returns_data)\naxes[1].plot(price_df['Date'][1:], cumulative_returns, linewidth=1, label='Cumulative Log Returns')\naxes[1].axvline(change_point_date, color='red', linestyle='--', linewidth=2, label=f'Detected Change Point: {change_point_date.strftime(\"%Y-%m-%d\")}')\naxes[1].set_title('Cumulative Log Returns with Detected Change Point', fontsize=14, fontweight='bold')\naxes[1].set_xlabel('Date')\naxes[1].set_ylabel('Cumulative Log Return')\naxes[1].legend()\naxes[1].grid(True, alpha=0.3)\n\n# Add nearby events\nfor _, event in nearby_events.iterrows():\n    axes[1].axvline(event['Date'], color='orange', alpha=0.5, linestyle=':')\n\nplt.tight_layout()\nplt.savefig('../reports/task2_change_point_visualization.png', dpi=300, bbox_inches='tight')\nplt.show()

## Conclusions and Interpretation\n\n### Key Findings:\n1. **Detected Change Point**: The Bayesian model identified a significant structural break in the Brent oil price return series around the estimated date.\n2. **Statistical Impact**: The change in mean returns before and after the break point quantifies the magnitude of the structural shift.\n3. **Event Association**: Major geopolitical or economic events near the detected change point provide context for the observed structural break.\n\n### Model Limitations:\n1. **Single Change Point**: This model assumes only one structural change, while real-world data may have multiple change points.\n2. **Mean Shift Only**: The model only captures changes in mean, not changes in volatility or higher-order moments.\n3. **Independence Assumption**: The model assumes returns are independent, ignoring potential autocorrelation and volatility clustering.\n\n### Advanced Modeling Approaches:\n1. **Multiple Change Points**: Extend the model to detect multiple structural breaks using methods like Bayesian online change point detection or hidden Markov models.\n2. **Volatility Change Points**: Model changes in both mean and volatility using stochastic volatility models.\n3. **Time-Varying Parameters**: Use state-space models with time-varying parameters to capture gradual changes.\n4. **Regime-Switching Models**: Implement Markov regime-switching models to capture different market regimes.\n5. **Incorporate Exogenous Variables**: Include event indicators, economic variables, or sentiment measures as covariates.\n\n### Practical Implications:\n- **Risk Management**: Understanding structural breaks helps in adjusting risk models and hedging strategies.\n- **Trading Strategies**: Change point detection can inform strategy adjustments around major market shifts.\n- **Policy Analysis**: Associating change points with specific events provides insights into market impacts of policy decisions.\n\n## Next Steps:\n1. Implement multiple change point detection\n2. Develop volatility change point models\n3. Create real-time change point detection systems\n4. Integrate fundamental analysis with statistical change point detection